# Column 04 — Donchian and Turtle breakouts with trend, cycle and volume confirmation

Classical Donchian/Turtle logic enters on a price breakout. The DeTime rewrite keeps the breakout trigger, but requires the move to agree with decomposed trend, cycle timing, residual overextension and volume participation.

The point is to reduce false breakouts, not to hide the original strategy.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from examples.quant_trading.data import load_sample_goog_ohlcv, market_data_manifest, ohlcv_audit_report
from examples.quant_trading.features import build_feature_table, walkforward_decompose_ohlcv
from examples.quant_trading.classic_indicators import donchian_channels
from examples.quant_trading.strategy_breakout import (
    breakout_diagnostic_table,
    compare_breakout_suites,
    make_classic_breakout_weight_grid,
    make_detime_breakout_weight_grid,
    run_classical_breakout_baselines,
    run_detime_breakout_baselines,
)
from examples.quant_trading.validation import compare_weight_strategies, turnover_report, write_run_audit, write_run_manifest

pd.set_option("display.max_columns", 30)
report_dir = Path("examples/quant_trading/reports")
report_dir.mkdir(parents=True, exist_ok=True)


## 1. Load real OHLCV data

Breakout logic needs high and low, not only close. The offline sample is still real historical market data.


In [ ]:
ohlcv_single = load_sample_goog_ohlcv(trim_start="2014-01-01")
ticker = ohlcv_single.attrs.get("symbol", "GOOG")
ohlcv = {field: ohlcv_single[[field]].rename(columns={field: ticker}) for field in ["Open", "High", "Low", "Close", "Volume"]}
prices = ohlcv["Close"]
manifest = market_data_manifest(
    tickers=[ticker],
    start=str(prices.index.min().date()),
    end=str(prices.index.max().date()),
    interval="1d",
    source=ohlcv_single.attrs.get("source", "bundled real sample"),
)
audit = ohlcv_audit_report(ohlcv)
display(audit)


## 2. Classical Donchian / Turtle baselines

The baselines include a 20/10 channel, a 55/20 Turtle-style channel, an ATR-capped Turtle variant and a simple volatility breakout.


In [ ]:
classic_weights = make_classic_breakout_weight_grid(ohlcv, allow_short=False)
classic_table, classic_results = compare_weight_strategies(prices, classic_weights, fee_bps=1.0, slippage_bps=2.0)
display(classic_table[["total_return", "cagr", "sharpe", "max_drawdown", "average_turnover"]].round(4))


In [ ]:
channels = donchian_channels(ohlcv["High"], ohlcv["Low"], entry_window=55, exit_window=20)
fig, ax = plt.subplots(figsize=(10, 4))
prices[ticker].plot(ax=ax, linewidth=1.0, label="close")
channels.upper_entry[ticker].plot(ax=ax, linewidth=0.9, label="55-day upper entry")
channels.lower_exit[ticker].plot(ax=ax, linewidth=0.9, label="20-day lower exit")
ax.set_title("Classical Donchian/Turtle channel")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## 3. Add DeTime trend, cycle, residual and volume context

A breakout is more credible when price trend is rising, cycle timing is not fighting the move, residual is not already extremely overextended and volume confirms participation.


In [ ]:
features = walkforward_decompose_ohlcv(
    ohlcv,
    method="STL",
    period="auto",
    period_candidates=(21, 42, 63, 126),
    train_window=252,
    step=63,
)
feature_tail = build_feature_table(prices, features).tail(120)
display(feature_tail.tail(5).round(4))
feature_tail.to_csv(report_dir / "column_04_feature_table_tail.csv")


In [ ]:
diagnostics = breakout_diagnostic_table(ohlcv, features, entry_window=55, exit_window=20)
display(diagnostics.round(4))
diagnostics.to_csv(report_dir / "column_04_breakout_diagnostics.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 3.5))
features["trend_slope"][ticker].plot(ax=ax, linewidth=1.0, label="trend slope")
features["cycle_slope"][ticker].plot(ax=ax, linewidth=1.0, label="cycle slope")
features["volume_residual_z"][ticker].plot(ax=ax, linewidth=0.8, alpha=0.7, label="volume residual z")
features["residual_z"][ticker].plot(ax=ax, linewidth=0.8, alpha=0.7, label="price residual z")
ax.axhline(0, linewidth=0.8)
ax.set_title("Breakout confirmation features")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## 4. DeTime breakout strategies

The DeTime variants keep the Donchian trigger, then add confirmation gates. One ablation removes cycle confirmation so the notebook can show what the cycle gate changes.


In [ ]:
detime_weights = make_detime_breakout_weight_grid(ohlcv, features, allow_short=False)
detime_table, detime_results = compare_weight_strategies(prices, detime_weights, fee_bps=1.0, slippage_bps=2.0)
display(detime_table[["total_return", "cagr", "sharpe", "max_drawdown", "average_turnover"]].round(4))


In [ ]:
classical = run_classical_breakout_baselines(ohlcv, allow_short=False, fee_bps=1.0, slippage_bps=2.0)
detime = run_detime_breakout_baselines(ohlcv, features, allow_short=False, fee_bps=1.0, slippage_bps=2.0)
comparison = compare_breakout_suites(classical, detime)
display(comparison[["strategy_group", "cagr", "sharpe", "max_drawdown", "average_turnover", "hit_rate"]].round(4))
comparison.to_csv(report_dir / "column_04_strategy_comparison.csv")
turnover_report({**classic_weights, **detime_weights}).to_csv(report_dir / "column_04_turnover_report.csv")
write_run_audit(report_dir, data_manifest=manifest, audit=audit, strategy_stats=comparison, prefix="column_04")
write_run_manifest(
    report_dir / "column_04_run_manifest.json",
    command="notebook:04_donchian_turtle_breakout_volume_confirmation",
    dataset="bundled_real_GOOG",
    strategies=list(comparison.index),
    result_file=str(report_dir / "column_04_strategy_comparison.csv"),
)


In [ ]:
chosen = "detime_donchian_55_20_trend_cycle_volume"
fig, ax1 = plt.subplots(figsize=(10, 4))
prices[ticker].plot(ax=ax1, linewidth=1.0, label="close")
channels.upper_entry[ticker].plot(ax=ax1, linewidth=0.8, alpha=0.7, label="entry channel")
ax2 = ax1.twinx()
detime_weights[chosen][ticker].plot(ax=ax2, linewidth=0.9, alpha=0.7, label="position")
ax1.set_title(f"Position overlay: {chosen}")
ax2.set_ylabel("position")
ax1.grid(True, alpha=0.3)
plt.show()


## Takeaway

Breakout systems are trend-following systems with an execution trigger. DeTime gives the trigger context: trend slope says whether the direction is structurally supported, cycle slope says whether timing is adverse, residual z-score says whether the move is already stretched, and volume decomposition checks participation.
